<a href="https://colab.research.google.com/github/liu-kristina/NLP/blob/main/Predicting_E_Commerce_Product_Recommendation_from_Reviews.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>



# <h1 align="center"> Predicting E-Commerce Product Recommendations from Reviews </h1> </center>

<p style="margin-bottom:1cm;"></p>

_____




![](https://github.com/dipanjanS/feature_engineering_session_dhs18/blob/master/ecommerce_product_ratings_prediction/clothing_banner.jpg?raw=1)

This is a classic NLP problem dealing with data from an e-commerce store focusing on women's clothing. Each record in the dataset is a customer review which consists of the review title, text description and a recommendation 0 or 1) for a product amongst other features

# Load up basic dependencies

In [1]:
import numpy as np
import pandas as pd
from sklearn.metrics import classification_report, confusion_matrix
import nltk
nltk.download('punkt')
nltk.download('stopwords')
nltk.download('averaged_perceptron_tagger')

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt.zip.
[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.
[nltk_data] Downloading package averaged_perceptron_tagger to
[nltk_data]     /root/nltk_data...
[nltk_data]   Unzipping taggers/averaged_perceptron_tagger.zip.


True

In [2]:
!pip install contractions
!pip install textsearch
!pip install tqdm

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 345.1/345.1 kB 6.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 114.9/114.9 kB 9.7 MB/s eta 0:00:00


# Load and View the Dataset

The data is available at https://www.kaggle.com/nicapotato/womens-ecommerce-clothing-reviews.

In [3]:
df = pd.read_csv('https://github.com/dipanjanS/text-analytics-with-python/raw/master/media/Womens%20Clothing%20E-Commerce%20Reviews%20-%20NLP.csv', keep_default_na=False)
df.head()

,Clothing ID,Age,Title,Review Text,Rating,Recommended IND,Positive Feedback Count,Division Name,Department Name,Class Name
0,767,33,,Absolutely wonderful - silky and sexy and comf...,4,1,0,Initmates,Intimate,Intimates
1,1080,34,,Love this dress! it's sooo pretty. i happene...,5,1,4,General,Dresses,Dresses
2,1077,60,Some major design flaws,I had such high hopes for this dress and reall...,3,0,0,General,Dresses,Dresses
3,1049,50,My favorite buy!,"I love, love, love this jumpsuit. it's fun, fl...",5,1,0,General Petite,Bottoms,Pants
4,847,47,Flattering shirt,This shirt is very flattering to all due to th...,5,1,6,General,Tops,Blouses


# Basic Data Processing

- Merge all review text attributes (title, text description) into one attribute
- Subset out columns of interest

In [4]:
df['Review'] = (df['Title'].map(str) +' '+ df['Review Text']).apply(lambda row: row.strip())
df['Recommended'] = df['Recommended IND']
df = df[['Review', 'Recommended']]
df.head()

,Review,Recommended
0,Absolutely wonderful - silky and sexy and comf...,1
1,Love this dress! it's sooo pretty. i happene...,1
2,Some major design flaws I had such high hopes ...,0
3,"My favorite buy! I love, love, love this jumps...",1
4,Flattering shirt This shirt is very flattering...,1


## Remove all records with no review text

In [5]:
df = df[df['Review'] != '']
df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 22642 entries, 0 to 23485
Data columns (total 2 columns):
 #   Column       Non-Null Count  Dtype 
---  ------       --------------  ----- 
 0   Review       22642 non-null  object
 1   Recommended  22642 non-null  int64 
dtypes: int64(1), object(1)
memory usage: 530.7+ KB


## There is some imbalance in the data based on product recommendations

In [6]:
df['Recommended'].value_counts()

,count
Recommended,
1,18541
0,4101


# Build train and test datasets

In [7]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(df.drop(columns=['Recommended']), df['Recommended'], test_size=0.3, random_state=42)
X_train.shape, X_test.shape

((15849, 1), (6793, 1))

In [8]:
from collections import Counter
Counter(y_train), Counter(y_test)

(Counter({1: 12966, 0: 2883}), Counter({1: 5575, 0: 1218}))

In [9]:
X_train.head(3)

,Review
4654,Sexy and flows I love this jumpsuit! i'm usual...
5333,Wanted to love it The dresss is much shorter t...
22502,So cute! though inside not soft I got the crea...


In [10]:
y_train[:3]

,Recommended
4654,1
5333,0
22502,1


# Text Pre-processing and Wrangling

We want to extract some specific features based on standard NLP feature engineering models like the classic Bag of Words model.
For this we need to clean and pre-process our text data. We will build a simple text pre-processor here since the main intent is to look at feature engineering strategies.

We will focus on:
- Text Lowercasing
- Removal of contractions
- Removing unnecessary characters, numbers and symbols
- Stopword removal

Source: https://towardsdatascience.com/a-practitioners-guide-to-natural-language-processing-part-i-processing-understanding-text-9f4abfd13e72

In [16]:
import nltk
nltk.download('punkt_tab')
import contractions
import re
import tqdm

# remove some stopwords to capture negation in n-grams if possible
stopwords = nltk.corpus.stopwords.words('english')
stopwords.remove('no')
stopwords.remove('not')
stopwords.remove('but')


def normalize_document(doc):
    # fix contractions
    doc = contractions.fix(doc)
    # remove special characters and digits
    doc = re.sub(r'[^a-zA-Z\s]', '', doc, flags=re.I|re.A)
    # lower case
    doc = doc.lower()
    # strip whitespaces
    doc = doc.strip()
    # tokenize document
    tokens = nltk.word_tokenize(doc)
    #filter stopwords out of document
    filtered_tokens = [token for token in tokens if token not in stopwords]
    # re-create document from filtered tokens
    doc = ' '.join(filtered_tokens)
    return doc

def normalize_corpus(docs):
    norm_docs = []
    for doc in tqdm.tqdm(docs):
        norm_doc = normalize_document(doc)
        norm_docs.append(norm_doc)

    return norm_docs


[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


In [17]:
X_train['Clean Review'] = normalize_corpus(X_train['Review'].values)
X_test['Clean Review'] = normalize_corpus(X_test['Review'].values)

100%|██████████| 6793/6793 [00:02<00:00, 2709.48it/s]


# Experiment 1: Modeling based on Bag of Words based Features - 1-grams

This is perhaps the most simple vector space representational model for unstructured text. A vector space model is simply a mathematical model to represent unstructured text (or any other data) as numeric vectors, such that each dimension of the vector is a specific feature\attribute.

The bag of words model represents each text document as a numeric vector where each dimension is a specific word from the corpus and the value could be its frequency in the document, occurrence (denoted by 1 or 0) or even weighted values.

The model’s name is such because each document is represented literally as a ‘bag’ of its own words, disregarding word orders, sequences and grammar.

Source: https://towardsdatascience.com/understanding-feature-engineering-part-3-traditional-methods-for-text-data-f6f7d70acd41

## Use the following default config for count vectorizer

- `min_df` as 0.0
- `max_df` as 1.0
- `ngram_range` as (1,1)

In [18]:
from sklearn.feature_extraction.text import CountVectorizer

cv = CountVectorizer(min_df=0.0, max_df=1.0, ngram_range=(1, 1))

X_traincv = cv.fit_transform(X_train['Clean Review'])
X_testcv = cv.transform(X_test['Clean Review'])

In [19]:
X_traincv

<Compressed Sparse Row sparse matrix of dtype 'int64'
	with 445720 stored elements and shape (15849, 15483)>

## Model Training and Evaluation

Train a logistic regression model and check the performance on the test data using the confusion matrix and classification report

In [20]:
from sklearn.linear_model import LogisticRegression

lr = LogisticRegression(random_state=42, solver='lbfgs', max_iter=1000)

In [21]:
lr.fit(X_traincv, y_train)
predictions = lr.predict(X_testcv)

print(confusion_matrix(y_test, predictions))
print(classification_report(y_test, predictions))

[[ 799  419]
 [ 254 5321]]
              precision    recall  f1-score   support

           0       0.76      0.66      0.70      1218
           1       0.93      0.95      0.94      5575

    accuracy                           0.90      6793
   macro avg       0.84      0.81      0.82      6793
weighted avg       0.90      0.90      0.90      6793



# Experiment 2: Modeling with Bag of Words based Features - 2-grams

We use the same feature engineering technique here except we consider both 1 and 2-grams as our features.

In [29]:
cv = CountVectorizer(min_df=0.0, max_df=1.0, ngram_range=(1, 2))

X_traincv = cv.fit_transform(X_train['Clean Review'])
X_testcv = cv.transform(X_test['Clean Review'])

In [30]:
X_traincv

<Compressed Sparse Row sparse matrix of dtype 'int64'
	with 930010 stored elements and shape (15849, 227644)>

## Model Training and Evaluation

In [23]:
lr.fit(X_traincv, y_train)
predictions = lr.predict(X_testcv)

print(confusion_matrix(y_test, predictions))
print(classification_report(y_test, predictions))

[[ 819  399]
 [ 215 5360]]
              precision    recall  f1-score   support

           0       0.79      0.67      0.73      1218
           1       0.93      0.96      0.95      5575

    accuracy                           0.91      6793
   macro avg       0.86      0.82      0.84      6793
weighted avg       0.91      0.91      0.91      6793



Some minor improvements

# Experiment 3: Adding Bag of Words based Features - 3-grams

We use the same feature engineering technique here except we consider 1, 2 and 3-grams as our features.

In [33]:
cv = CountVectorizer(min_df=0.0, max_df=1.0, ngram_range=(1, 3))

X_traincv = cv.fit_transform(X_train['Clean Review'])
X_testcv = cv.transform(X_test['Clean Review'])

In [34]:
X_traincv

<Compressed Sparse Row sparse matrix of dtype 'int64'
	with 1401913 stored elements and shape (15849, 640475)>

## Model Training and Evaluation

In [25]:
lr.fit(X_traincv, y_train)
predictions = lr.predict(X_testcv)

print(confusion_matrix(y_test, predictions))
print(classification_report(y_test, predictions))

[[ 838  380]
 [ 234 5341]]
              precision    recall  f1-score   support

           0       0.78      0.69      0.73      1218
           1       0.93      0.96      0.95      5575

    accuracy                           0.91      6793
   macro avg       0.86      0.82      0.84      6793
weighted avg       0.91      0.91      0.91      6793



# Experiment 4: Adding Bag of Words based Features - 3-grams with Feature Selection

Set `min_df` as 3 in CountVectorizer and keep other params same as the previous experiment and notice the drop in features.

We drop all words \ n-grams which occur less than 3 times in all documents.


In [26]:
cv = CountVectorizer(min_df=3, max_df=1., ngram_range=(1, 3))

X_traincv = cv.fit_transform(X_train['Clean Review'])
X_testcv = cv.transform(X_test['Clean Review'])

In [28]:
X_traincv

<Compressed Sparse Row sparse matrix of dtype 'int64'
	with 762164 stored elements and shape (15849, 44455)>

## Model Training and Evaluation

In [27]:
lr.fit(X_traincv, y_train)
predictions = lr.predict(X_testcv)

print(confusion_matrix(y_test, predictions))
print(classification_report(y_test, predictions))

[[ 838  380]
 [ 234 5341]]
              precision    recall  f1-score   support

           0       0.78      0.69      0.73      1218
           1       0.93      0.96      0.95      5575

    accuracy                           0.91      6793
   macro avg       0.86      0.82      0.84      6793
weighted avg       0.91      0.91      0.91      6793



Similar results with much less features!

# Experiment 5: Modeling on FastText Averaged Document Embeddings

## Build the FastText embedding model here


__10 epochs might take 5 mins on Colab__

In [36]:
pip install gensim

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 27.9/27.9 MB 22.6 MB/s eta 0:00:00


In [37]:
%%time

from gensim.models import FastText

tokenized_docs_train = [doc.split() for doc in X_train['Clean Review']]
# sample config params vector_size: 300, window: 30, min_count=2 or more, epochs=10
ft_model = FastText(tokenized_docs_train, vector_size=300, window=30, min_count=2, workers=4, sg=1, epochs=10)

CPU times: user 8min 49s, sys: 2.88 s, total: 8min 52s
Wall time: 6min 4s


## Generate document level embeddings

Word embedding models give us an embedding for each word, how can we use it for downstream ML\DL tasks? one way is to flatten it or use sequential models. A simpler approach is to average all word embeddings for words in a document and generate a fixed-length document level emebdding

In [38]:
def average_word_vectors(words, model, vocabulary, num_features):

    feature_vector = np.zeros((num_features,),dtype="float64") # avg embedding for the tokenized doc ['sky', 'blue', 'beautiful']
    nwords = 0.

    for word in words: # for every word in a document, find the word embedding and add it to feature_vector
        if word in vocabulary:
            nwords = nwords + 1.    # 1 + 1 + 1 = 3
            feature_vector = np.add(feature_vector, model.wv[word])
            # model.wv['sky'] + model.wv['blue'] + model.wv['beautiful'] = sum(3x15) = 1x15

    if nwords:
        feature_vector = np.divide(feature_vector, nwords)  # 1x15 / 3

    return feature_vector


def averaged_word_vectorizer(corpus, model, num_features):
    vocabulary = set(model.wv.index_to_key) # set of unique words across the corpus
    features = [average_word_vectors(tokenized_sentence, model, vocabulary, num_features)
                    for tokenized_sentence in corpus]
    return np.array(features)

In [40]:
tokenized_docs_train = [doc.split() for doc in X_train['Clean Review']]
tokenized_docs_test = [doc.split() for doc in X_test['Clean Review']]

Xtrain_doc_vecs_ft = averaged_word_vectorizer(tokenized_docs_train, ft_model, 300)
Xtest_doc_vecs_ft = averaged_word_vectorizer(tokenized_docs_test, ft_model, 300)

Xtrain_doc_vecs_ft.shape, Xtest_doc_vecs_ft.shape

((15849, 300), (6793, 300))

## Model Training and Evaluation

In [41]:
lr.fit(Xtrain_doc_vecs_ft, y_train)
predictions = lr.predict(Xtest_doc_vecs_ft)

print(confusion_matrix(y_test, predictions))
print(classification_report(y_test, predictions))

[[ 743  475]
 [ 170 5405]]
              precision    recall  f1-score   support

           0       0.81      0.61      0.70      1218
           1       0.92      0.97      0.94      5575

    accuracy                           0.91      6793
   macro avg       0.87      0.79      0.82      6793
weighted avg       0.90      0.91      0.90      6793

